# Meeting with Quentin, 10 Nov 2026

Key takeaways: <br>

- Focus on training set for angles to see if I can get it to 'overtrain' such that we know that my model is expressive enough. 
    - **I think I should try to predict $\sin(\theta),\cos(\theta)$ instead of $\theta$ directly** . Predicting $\theta$ directly could be a reason why large angles are poorly represented ($-180^\circ$ and $175^\circ$ are actually very close ($5^\circ$ apart) but are seen as $355^\circ$ away and the model is heavily penalized)

- Revert back to 1 model for two target variables (labels), i.e. instead of 1 NN for u and v, or 1 NN for norms, another NN for angles, I should train 1 model for Angle and Norm.  
    - *Unsure how deep that network should be; I think $256\rightarrow 128\rightarrow 64 $ is over-expressive for the Norm but I don't know for the angles. Indeed, in the picture below, we see that using the same predictors(inputs/features), loss function and training lenght the predictions are better for the complex (256) model rather than the simpler (simplr) model*<br>
<img src=".\attachements\SimplrAngles.png" alt="drawing" width="50%"/> <br>
I also have the models written below

-  Focus on the physics equations, i.e. instead of the Loss functions: $L$ being 
$$L= \sqrt{\frac{1}{n_{batch}}\sum^{n_{batch}}(u_d^i-u_\theta^i)^2} \text{, where }\theta \text{ represent the model, } d \text{ stand for data, and } n_{batch} \text{ is the batch size}$$
We would implement, **without writing the root mean and square**
$$L= u_d^i-(A_\theta^au^a+u_w) \text{ where } A \text{ is a complex coefficient: } |A|e^{i\theta}$$
and/or
$$L= u_d^i-(\sqrt{\frac{\rho^aC_d^a}{\rho^wC_d^w}}e^{i\theta}u^a+u_w) \text{ where we predict  }C_d^a \text{, }C_d^w \text{, and }\theta$$
I would probably need to do some (L1/L2) regularization to avoid C_d becoming too large and unrealistic (how exactly is an intersting discussion), and probably some penality for the angles of the form $\cos^2+\sin^2-1$ <br>
Even maybe one day a more complete momentum equation including the internal stress term.

- Focus on the relevant inputs not more, namely the ones present in the (linearized) Momentum Equation

In [ ]:
import torch.nn as nn
import torch.nn.functional  as F
import torch
import numpy as np
class NNforNormsandAngles(nn.Module):
    """ MLP with Relu, 256->128->64\n
    Input: n_features (torch.tensor)\n
    Outputs: F.softplus(norm),cos, sin #(all tensors)\n
    The final layer is linear for the angles BUT NOT FOR the Norms (softplus= ln(1+e^x))\n
    Note that in normalizing the data, the norms are normalized by Log1P (so the output norms need to be Exp1m to be interpreted at inference)\n
    the cos/sin are already between -1 and 1.\n
         """
    def __init__(self, n_inputs):
        super().__init__()
        self.fc1=nn.Linear(n_inputs,256)
        self.fc2=nn.Linear(256,128)
        self.fc3=nn.Linear(128,64)
        self.final=nn.Linear(64,3)


    def forward(self,x):
        x=F.relu(self.fc1(x))
        x=F.relu(self.fc2(x))
        x=F.relu(self.fc3(x))

        x=self.final(x) # (batch, 3)
        norm = x[:,0]   # (batch,)
        cos = x[:,1]    # (batch,)
        sin = x[:,2]    # (batch,)
        return F.softplus(norm),cos, sin


tensor([-1.])
